# Notebook 01 — Exploração de Benchmarks de Arquiteturas de Visão

## Objetivos

1. Carregar e explorar o dataset de benchmarks de arquiteturas de visão computacional
2. Analisar trade-offs entre CNNs, Vision Transformers e arquiteturas híbridas
3. Visualizar o impacto da resolução de entrada nos FLOPs e latência
4. Identificar padrões de eficiência e pontos na fronteira de Pareto

## Fundamentação Teórica

Conforme discutido na seção *Fundamentos de Complexidade Computacional* do material:

> A complexidade de operações convolucionais é dada por:
> $$\text{FLOPs}_{\text{conv}} = 2 \times K^2 \times C_{in} \times C_{out} \times H_{out} \times W_{out}$$
>
> Para Vision Transformers, o mecanismo de self-attention apresenta complexidade quadrática:
> $$\text{FLOPs}_{\text{attention}} = 4 \times n \times d^2 + 2 \times n^2 \times d$$
>
> — He & Sun (2015); Dosovitskiy et al. (2021)

A escolha de arquitetura impacta diretamente latência, throughput e custo operacional.
Neste notebook, exploramos dados de benchmark para entender esses trade-offs.

In [ ]:
# Imports
import sys
import os

# Adicionar raiz do projeto ao path
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_preprocessing import DataPreprocessor
from src.evaluation import plot_pareto_frontier, plot_resolution_impact, plot_family_comparison

# Configurações de visualização
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 15)

print('Setup completo.')

## 1. Geração e Carregamento do Dataset

O dataset contém ~5000 registros de benchmark para 20 arquiteturas de visão,
variando resolução de entrada (128–512), hardware (V100, RTX 3090, CPU) e
incluindo múltiplas execuções para robustez estatística.

In [ ]:
# Gerar dataset se não existir
dp = DataPreprocessor(random_state=42)

dataset_path = '../data/raw/dataset.csv'
if not os.path.exists(dataset_path):
    print('Gerando dataset de benchmarks...')
    dp.generate_benchmark_dataset(dataset_path)

# Carregar dataset
df = dp.load_data(dataset_path)
print(f'Dataset carregado: {df.shape[0]} registros, {df.shape[1]} colunas')
df.head(10)

## 2. Visão Geral do Dataset

Examinamos a distribuição de modelos, famílias, resoluções e hardwares.

In [ ]:
# Estatísticas gerais
print('=== Estatísticas Descritivas ===')
print(df.describe().round(2))
print(f'\nModelos únicos: {df["model_name"].nunique()}')
print(f'Famílias: {df["model_family"].unique().tolist()}')
print(f'Resoluções: {sorted(df["input_resolution"].unique())}')
print(f'Hardwares: {df["hardware"].unique().tolist()}')

In [ ]:
# Distribuição de registros por família e hardware
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Por família
family_counts = df['model_family'].value_counts()
axes[0].bar(family_counts.index, family_counts.values,
            color=['#2196F3', '#4CAF50', '#F44336'])
axes[0].set_title('Registros por Família de Modelo')
axes[0].set_ylabel('Contagem')

# Por resolução
res_counts = df['input_resolution'].value_counts().sort_index()
axes[1].bar(res_counts.index.astype(str), res_counts.values, color='#FF9800')
axes[1].set_title('Registros por Resolução')
axes[1].set_ylabel('Contagem')

# Por hardware
hw_counts = df['hardware'].value_counts()
axes[2].bar(hw_counts.index, hw_counts.values, color='#9C27B0')
axes[2].set_title('Registros por Hardware')
axes[2].set_ylabel('Contagem')

plt.tight_layout()
plt.show()

## 3. Preparação de Dados Agregados

Agregamos múltiplas execuções por configuração para obter estatísticas robustas
(média, desvio-padrão, percentis), conforme Reddi et al. (2020) — múltiplas
execuções são essenciais para benchmarks reproduzíveis.

In [ ]:
# Agregar dados por configuração
agg_df = dp.prepare_features(df)
print(f'Dados agregados: {len(agg_df)} configurações únicas')
agg_df.head(10)

## 4. Análise de FLOPs: CNN vs ViT

Conforme seção 1 do material, CNNs têm complexidade O(K² × C_in × C_out × H × W)
enquanto ViTs apresentam complexidade O(n² × d) no mecanismo de atenção.

Isso resulta em perfis de custo computacional fundamentalmente diferentes.

In [ ]:
# Comparação de FLOPs por família (hardware V100, resolução 224)
subset_v100_224 = agg_df[
    (agg_df['hardware'] == 'v100') & (agg_df['input_resolution'] == 224)
].copy()

fig, ax = plt.subplots(figsize=(14, 6))
palette = {'cnn': '#2196F3', 'vit': '#F44336', 'hybrid': '#4CAF50'}

for family in ['cnn', 'vit', 'hybrid']:
    subset = subset_v100_224[subset_v100_224['model_family'] == family]
    ax.barh(
        subset['model_name'], subset['flops_giga'],
        color=palette[family], alpha=0.8, label=family.upper(),
    )

ax.set_xlabel('FLOPs (GFLOPs)', fontsize=12)
ax.set_title('FLOPs por Arquitetura (resolução 224×224, V100)', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

print('\n=== Tabela Comparativa (Resolução 224, V100) ===')
display_cols = ['model_name', 'model_family', 'params_millions', 'flops_giga',
                'top1_accuracy', 'latency_mean_ms']
print(subset_v100_224[display_cols].sort_values('flops_giga').to_string(index=False))

## 5. Impacto da Resolução nos FLOPs e Latência

Conforme seção 4 do material (Tabela 2):

> Dobrando-se a resolução de entrada (de 224×224 para 448×448), o número de FLOPs
> em camadas convolucionais aumenta aproximadamente por um fator de 4, dado que
> tanto H_out quanto W_out dobram.

Vejamos este efeito nos dados de benchmark.

In [ ]:
# Impacto da resolução para EfficientNet-B0 (V100)
fig = plot_resolution_impact(
    df, model_name='efficientnet_b0', hardware='v100',
    save_path='../outputs/figures/resolution_impact_effnet_b0.png',
)
plt.show()

# Calcular fator de crescimento real vs teórico
res_data = dp.get_resolution_impact(df, model_name='efficientnet_b0', hardware='v100')
res_agg = res_data.groupby('input_resolution')['flops_giga'].mean()
base_flops = res_agg.iloc[1]  # 224
print('\n=== Fator de Crescimento de FLOPs (relativo a 224×224) ===')
for res, flops in res_agg.items():
    theoretical = (res / 224) ** 2
    actual = flops / base_flops
    print(f'  {res}×{res}: teórico = {theoretical:.2f}x, medido = {actual:.2f}x')

## 6. Comparação CNN vs ViT vs Híbrido

Conforme seção 3 do material:

> DeiT-Small, com 22M parâmetros, alcança 79.8% top-1 em ImageNet,
> comparável a EfficientNet-B4 (82.9%) mas com perfil de latência distinto.
> Em GPUs modernas com Tensor Cores, ViTs podem apresentar latência
> competitiva, porém em edge devices, CNNs frequentemente mantêm vantagem.

In [ ]:
# Comparação de latência por família (V100)
v100_data = agg_df[agg_df['hardware'] == 'v100'].copy()

fig = plot_family_comparison(
    v100_data, metric='latency_mean_ms',
    save_path='../outputs/figures/family_latency_comparison.png',
)
plt.show()

# Estatísticas por família
print('\n=== Latência Média por Família (V100) ===')
family_stats = v100_data.groupby('model_family').agg(
    latency_mean=('latency_mean_ms', 'mean'),
    latency_std=('latency_mean_ms', 'std'),
    accuracy_mean=('top1_accuracy', 'mean'),
    n_architectures=('model_name', 'nunique'),
).round(2)
print(family_stats)

## 7. Scatter Plot: Acurácia vs Latência

Visualização central para a análise de trade-offs. Cada ponto representa
uma arquitetura em uma configuração específica de resolução.

In [ ]:
# Scatter: acurácia vs latência (V100, resolução 224)
fig, ax = plt.subplots(figsize=(14, 9))
palette = {'cnn': '#2196F3', 'vit': '#F44336', 'hybrid': '#4CAF50'}
markers = {'cnn': 'o', 'vit': 's', 'hybrid': 'D'}

for family in ['cnn', 'vit', 'hybrid']:
    subset = subset_v100_224[subset_v100_224['model_family'] == family]
    ax.scatter(
        subset['latency_mean_ms'], subset['top1_accuracy'],
        s=subset['params_millions'] * 5, alpha=0.7,
        color=palette[family], marker=markers[family],
        label=f'{family.upper()} (tamanho = parâmetros)',
        edgecolors='white', linewidth=0.5,
    )
    # Anotar modelos
    for _, row in subset.iterrows():
        ax.annotate(
            row['model_name'], (row['latency_mean_ms'], row['top1_accuracy']),
            textcoords='offset points', xytext=(5, 5), fontsize=8, alpha=0.8,
        )

ax.set_xlabel('Latência Média (ms)', fontsize=12)
ax.set_ylabel('Acurácia Top-1 (%)', fontsize=12)
ax.set_title('Trade-off Acurácia vs Latência — V100 @ 224×224', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/accuracy_vs_latency.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Parâmetros vs Eficiência

Conforme seção 2 do material, depthwise separable convolutions (MobileNet)
reduzem o número de operações sem perda proporcional de acurácia:

> Redução = 1/C_out + 1/K² ≈ 8-9x para K=3

In [ ]:
# Eficiência: acurácia por milhão de parâmetros
subset_v100_224['efficiency'] = (
    subset_v100_224['top1_accuracy'] / subset_v100_224['params_millions']
)

fig, ax = plt.subplots(figsize=(14, 6))
sorted_data = subset_v100_224.sort_values('efficiency', ascending=True)
colors = [palette[f] for f in sorted_data['model_family']]
ax.barh(sorted_data['model_name'], sorted_data['efficiency'], color=colors, alpha=0.8)
ax.set_xlabel('Eficiência (Acurácia % / Parâmetros M)', fontsize=12)
ax.set_title('Eficiência de Parâmetros por Arquitetura', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')

# Legenda manual
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=f.upper()) for f, c in palette.items()]
ax.legend(handles=legend_elements, fontsize=10)

plt.tight_layout()
plt.show()

print('\nTop-5 arquiteturas mais eficientes (acurácia/parâmetro):')
print(sorted_data[['model_name', 'model_family', 'params_millions',
                    'top1_accuracy', 'efficiency']].tail(5).to_string(index=False))

## 9. Correlação entre Métricas

Analisamos correlações entre FLOPs, latência, parâmetros e acurácia para
entender interdependências.

In [ ]:
# Matriz de correlação
corr_cols = ['params_millions', 'flops_giga', 'top1_accuracy',
             'latency_mean_ms', 'throughput_mean', 'memory_mean_mb']
corr_matrix = v100_data[corr_cols].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_matrix, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, square=True, ax=ax,
    xticklabels=[c.replace('_', ' ').title() for c in corr_cols],
    yticklabels=[c.replace('_', ' ').title() for c in corr_cols],
)
ax.set_title('Correlação entre Métricas de Performance (V100)', fontsize=14)
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 10. Análise de Hardware

Comparação de latência entre GPU V100, RTX 3090 e CPU para a mesma arquitetura.
Conforme seção 10 do material, a escolha de hardware restringe as opções na
fronteira de Pareto.

In [ ]:
# Latência por hardware para modelos selecionados (resolução 224)
selected_models = ['efficientnet_b0', 'resnet50', 'deit_small', 'swin_tiny']
hw_subset = agg_df[
    (agg_df['model_name'].isin(selected_models)) &
    (agg_df['input_resolution'] == 224)
].copy()

fig, ax = plt.subplots(figsize=(12, 6))
hw_pivot = hw_subset.pivot_table(
    index='model_name', columns='hardware', values='latency_mean_ms',
)
hw_pivot.plot(kind='bar', ax=ax, color=['#2196F3', '#F44336', '#4CAF50'])
ax.set_ylabel('Latência (ms)', fontsize=12)
ax.set_title('Latência por Hardware — Modelos Selecionados @ 224×224', fontsize=14)
ax.legend(title='Hardware')
ax.grid(True, alpha=0.3, axis='y')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 11. Análise EfficientNet: Compound Scaling

Conforme seção 2 do material, EfficientNet utiliza compound scaling:

> $\text{depth} = \alpha^\phi$, $\text{width} = \beta^\phi$, $\text{resolution} = \gamma^\phi$
>
> Com α=1.2, β=1.1, γ=1.15 e restrição α × β² × γ² ≈ 2

In [ ]:
from src.model import calculate_efficientnet_scaling

# Demonstrar compound scaling para diferentes valores de phi
print('=== EfficientNet Compound Scaling (Tan & Le, 2019) ===')
print(f'{"φ":>4} | {"Depth":>8} | {"Width":>8} | {"Resolution":>10} | {"α×β²×γ²":>8}')
print('-' * 50)

phis = [0, 1, 2, 3, 4, 5, 6, 7]
scaling_results = []
for phi in phis:
    s = calculate_efficientnet_scaling(phi)
    scaling_results.append(s)
    print(f'{phi:>4} | {s["depth"]:>8.2f} | {s["width"]:>8.2f} | {s["resolution"]:>10.2f} | {s["constraint_value"]:>8.2f}')

# Plotar crescimento dos fatores
fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(phis, [s['depth'] for s in scaling_results], 'o-', label='Depth (α^φ)', color='#2196F3')
ax.plot(phis, [s['width'] for s in scaling_results], 's-', label='Width (β^φ)', color='#F44336')
ax.plot(phis, [s['resolution'] for s in scaling_results], 'D-', label='Resolution (γ^φ)', color='#4CAF50')
ax.set_xlabel('φ (coeficiente de scaling)', fontsize=12)
ax.set_ylabel('Fator de Scaling', fontsize=12)
ax.set_title('EfficientNet Compound Scaling — α=1.2, β=1.1, γ=1.15', fontsize=14)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Resumo

Neste notebook, exploramos:

1. **Dataset de benchmarks** com ~5000 registros cobrindo 20 arquiteturas
2. **FLOPs por família**: CNNs eficientes (MobileNet, EfficientNet) dominam em baixo FLOPs
3. **Impacto da resolução**: crescimento ≈ quadrático de FLOPs, confirmando O(r²)
4. **CNN vs ViT**: ViTs exigem mais FLOPs para acurácia comparável a baixas resoluções
5. **Eficiência**: MobileNets e EfficientViT oferecem melhor acurácia por parâmetro
6. **Correlações**: FLOPs e latência são altamente correlacionados em GPUs
7. **EfficientNet scaling**: compound scaling balanceia depth, width e resolution

**Próximo notebook:** [02_treinamento.ipynb](02_treinamento.ipynb) — Benchmarking ao vivo
e treinamento de modelos CNN vs depthwise separable CNN.